# Experiment 4 — Activation swap (interchange intervention)

Reproduction guidebook §7.

> **Question.** Does PANL carry **confidence-specific** information, or merely answer
> *content* that happens to correlate with confidence?
> **Key result.** Cross-confidence swaps shift confidence **directionally** across unrelated
> question–answer pairs, beyond same-confidence controls. Peak L26.

**Logic (§7.1).** Transplant the PANL residual stream from a **donor** trial into a
**recipient** trial, leaving the recipient's question and answer completely unchanged. If
PANL caches a representation of the reported sentiment, a cross-condition swap biases the
recipient toward the donor's report *directionally*. If PANL merely encodes content
features, swaps produce generic disruption that does not depend on the donor's report.

In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "vconf").is_dir())
sys.path.insert(0, str(ROOT))

from vconf import notebook as nb

from vconf import exp4_swap as E4
from vconf import metrics as M
from vconf import plotting
from vconf.results import summarize

cfg = nb.run_config("gemma-categorical")
print(nb.describe(cfg))
print("2x2 design:", E4.CONDITIONS, "| controls:", E4.CONTROL_CONDITIONS)

In [ ]:
loaded = nb.open_model(cfg)
trials, rendered = nb.build_trials(loaded, cfg)
trials = nb.graded(trials)
print(len(trials), "trials")

## The disjoint partition (§2.3.4)

Several sets must be **mutually disjoint**, partitioned once up front: the
**activation-collection set**, the **calibration set**, and the per-experiment **test sets**.
`nb.split_activation_holdout` makes the first cut; recipients and donors below are both drawn
from the holdout, never from the activation-collection set.

In [ ]:
(act_trials, act_rendered), (hold_trials, hold_rendered) = nb.split_activation_holdout(
    trials, rendered, cfg
)
print(f"activation-collection set: {len(act_trials)} trials")
print(f"holdout (recipients + donors): {len(hold_trials)} trials")
print("disjoint:", {t.qid for t in act_trials} & {t.qid for t in hold_trials} == set())

In [ ]:
# Donor activations must come from the same trials the donors are drawn from.
store = nb.activation_store(loaded, cfg, hold_rendered, trials=hold_trials,
                            positions=("PANL", "PANL+1", "CC"))
print(store.get(cfg.layers[-1], "PANL").shape, "donor activations")

## Recipients and length-matched donors (§7.3)

* The **same** high-confidence recipients are used in H→H and H→L, and the same low-confidence
  recipients in L→L and L→H — this makes the control a *within-trial* comparison.
* The low pool is small (N = 221 in the paper), so it is **sampled with replacement**.
* **Donor–recipient length matching is required**: donors are matched to recipients on
  tokenized question length and answer length using 10-quantile bins, otherwise an apparent
  "confidence transfer" could be a length-mismatch artifact.

In [ ]:
design, matching = E4.build_swap_design(
    hold_trials, hold_rendered, n=cfg.trial_counts["swap"], model_key=cfg.model_key,
    seed=cfg.seed,
)
quality = pd.DataFrame(matching).T
quality["paper question_bin_match"] = E4.MATCHING_TARGETS["question_bin_match"]
quality["paper answer_bin_match"] = str(E4.MATCHING_TARGETS["answer_bin_match"])
quality

Matching quality is bounded by the size of the pools it draws from. The paper matches
question-length bins in 100% of cases and answer-length bins in 94–100%, with mean |ΔL_Q| ≈
1.5–2.7 tokens and mean |ΔL_A| ≈ 0.3–0.5, out of hundreds of donors per band. A smaller run
has fewer low-confidence trials to draw on, so the cross-confidence cells match less well —
the table above reports exactly how much less, which is the point of reporting it: a residual
length mismatch is the main artifact that could masquerade as confidence transfer, and it has
to be visible rather than assumed away.

In [ ]:
print("same recipients in H->H and H->L:",
      np.array_equal(design["H->H"][0], design["H->L"][0]))
print("same recipients in L->L and L->H:",
      np.array_equal(design["L->L"][0], design["L->H"][0]))
for condition, (recipients, donors) in design.items():
    print(f"{condition}: {len(recipients)} recipients, "
          f"{len(set(donors.tolist()))} distinct donors")

## Running the 2×2 (§7.4)

For each (layer, position, condition): run the donor clean and cache its residual stream at
that (layer, position), then run the recipient with the donor activation **replacing** the
recipient's own, and compute all three metrics relative to the recipient's clean run.

In [ ]:
frame = E4.run_swap(loaded, hold_rendered, hold_trials, design, store, cfg,
                    positions=("PANL", "PANL+1", "CC"))
summary = summarize(frame)
summary[summary["position"] == "PANL"]

## Directionality is the payload (§7.5)

| Condition | Confidence change | Logit-diff change | Token change rate |
|---|---|---|---|
| **L→H** | ≈ **+0.21** | ≈ −1.2 | ≈ 37% |
| **H→L** | ≈ **−0.08…−0.10** | ≈ **−2.0** | ≈ 30% |
| H→H (control) | ≈ 0 | ≈ −1.1 | ≈ 15% |
| L→L (control) | ≈ 0 | ≈ −0.3 | ≈ 12% |

The critical comparison is **cross- minus same-confidence**, not cross- versus zero: some
disruption from *any* swap is expected, and that is exactly what H→H and L→L quantify.

In [ ]:
fig = plotting.position_panels(
    summary, "confidence_change", ("PANL", "PANL+1", "CC"),
    ylabel="Δ confidence (swapped − clean)",
    suptitle=f"Activation swap — {cfg.name}",
)
plotting.save_figure(fig, f"exp4-swap-confidence-{cfg.name}")

In [ ]:
fig = plotting.position_panels(
    summary, "token_changed", ("PANL", "PANL+1", "CC"),
    ylabel="first-token change rate", suptitle="Activation swap — first-token change rate",
)
plotting.save_figure(fig, f"exp4-swap-tokenchange-{cfg.name}")

In [ ]:
difference = E4.cross_minus_same(summary)
difference[difference["position"].isin(["PANL", "PANL+1"])]

In [ ]:
panl = summary[summary["position"] == "PANL"]
peak = int(panl.loc[panl["confidence_change_mean"].abs().idxmax(), "layer"])
print(f"PANL peak layer: {peak}  (paper: {E4.PAPER_TARGETS['peak_layer']})")

fig, ax = plt.subplots(figsize=(5.5, 3.4))
plotting.condition_bars(summary, "confidence_change", layer=peak, position="PANL",
                        paper=E4.PAPER_TARGETS, ax=ax, ylabel="Δ confidence")
plotting.save_figure(fig, f"exp4-swap-conditions-{cfg.name}")

In [ ]:
rows = []
for condition in E4.CONDITIONS:
    row = panl[(panl["layer"] == peak) & (panl["condition"] == condition)]
    if row.empty:
        continue
    rows.append({
        "condition": condition,
        "Δ confidence": round(float(row["confidence_change_mean"].iloc[0]), 4),
        "SEM": round(float(row["confidence_change_sem"].iloc[0]), 4),
        "paper Δ confidence": E4.PAPER_TARGETS[condition]["confidence_change"],
        "Δ logit diff": round(float(row["logit_diff_change_mean"].iloc[0]), 3),
        "paper Δ logit diff": E4.PAPER_TARGETS[condition]["logit_diff_change"],
        "token change rate": round(float(row["token_changed_mean"].iloc[0]), 3),
        "paper rate": E4.PAPER_TARGETS[condition]["token_change_rate"],
    })
pd.DataFrame(rows)

In [ ]:
def paired(condition, control, position="PANL"):
    at_peak = frame[(frame["layer"] == peak) & (frame["position"] == position)]
    a = at_peak[at_peak["condition"] == condition].sort_values("trial")["confidence_change"]
    b = at_peak[at_peak["condition"] == control].sort_values("trial")["confidence_change"]
    n = min(len(a), len(b))
    return M.paired_comparison(a.to_numpy()[:n], b.to_numpy()[:n])

for condition, control in (("L->H", "L->L"), ("H->L", "H->H")):
    stats = paired(condition, control)
    print(f"{condition} vs {control} at PANL L{peak}: "
          f"Δ = {stats['mean_difference']:+.3f} ± {stats['sem_difference']:.3f}, "
          f"p = {stats['t_p']:.3g}, d = {stats['cohens_d']:.2f}")

In [ ]:
lh = paired("L->H", "L->L")["mean_difference"]
hl = paired("H->L", "H->H")["mean_difference"]
control_effect = abs(float(summary[(summary["position"] == "PANL+1")]["confidence_change_mean"]).max()) \
    if not summary[summary["position"] == "PANL+1"].empty else 0.0
checks = {
    "Q6 L->H raises confidence beyond its same-confidence control": lh > 0,
    "Q6 H->L lowers confidence beyond its same-confidence control": hl < 0,
    "Q6 the two cross conditions move in opposite directions": lh > 0 > hl,
    "PANL+1 shows no comparable effect": abs(lh) > control_effect,
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

**Interpretation.** Directional, donor-dependent shifts cannot be produced by generic
content disruption, which rules out the alternative that PANL merely encodes content features
correlated with confidence — the recipient's content is untouched.

**Asymmetry is expected in some settings** (§7.5 item 4). On MMLU and in Magistral the L→H
direction dominates, because those confidence distributions are concentrated at the top and
create a ceiling for high-confidence recipients. Asymmetry is not a failed replication.